# XLNX 历史成分股价格来源预检

固定两次 Polygon/Massive 只读 GET；仅验证来源权限与历史身份，不读取 SOXX 结果。


In [1]:
import os,sys,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_xlnx_polygon_price_preflight_20261023';snap=root/'data/snapshots/stage1_xlnx_polygon_price_preflight_20261023'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
plan=json.loads((card/'plan.json').read_text());transport=json.loads((card/'transport_result.json').read_text())
assert plan['frozen'] and transport['plan_sha256']==sha(card/'plan.json') and len(transport['attempts'])==2 and transport['retries']==0
assert sha(root/plan['source_basis']['prior_member_price_gate'])==plan['source_basis']['prior_gate_sha256']
for row in transport['attempts']:
 assert row['method']=='GET' and row['endpoint_path']==plan['exact_requests'][row['sequence']-1]['path']
 assert row['query_without_credential']==plan['exact_requests'][row['sequence']-1]['query_without_credential']
 assert row['body_sha256']==sha(root/row['raw_path']) and row['body_bytes']==(root/row['raw_path']).stat().st_size
ref=json.loads((snap/'reference.json').read_text());agg=json.loads((snap/'aggregates.json').read_text())
identity={k:ref['results'].get(k) for k in ('ticker','name','cik','composite_figi','share_class_figi','list_date','active','type','market')}
assert transport['attempts'][0]['http_status']==200 and ref['status']=='OK' and identity['ticker']=='XLNX' and identity['name']=='Xilinx Inc' and identity['market']=='stocks'
assert transport['attempts'][1]['http_status']==403 and agg['status']=='NOT_AUTHORIZED' and 'timeframe' in agg['message']
report={'decision':'current_Polygon_plan_XLNX_history_not_authorized_source_identity_only','plan_sha256':sha(card/'plan.json'),'reference_HTTP':200,'aggregate_HTTP':403,'provider_reference_status':ref['status'],'provider_aggregate_status':agg['status'],'provider_aggregate_message':agg['message'],'asof_2020_identity':identity,'historical_daily_price_rows':0,'2019_2022_constituent_price_source_qualified':False,'historical_first_receipt_qualified':False,'SOXX_outcomes_read':0,'new_model_fits':0,'new_native_accounts':0,'HTTP_GETs':2,'retries':0}
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
settings=_settings(root);members=[card/'plan.json',card/'transport_result.json',snap/'reference.json',snap/'aggregates.json']
with tempfile.TemporaryDirectory(prefix='xlnx-source-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for p in members:out.add(p,arcname=str(p.relative_to(root)))
 digest=sha(archive)
 response=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')
 receipt=response['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 nas_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 restored=Path(tmp)/'restored.tar.gz'
 with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(nas_path)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()}
  assert set(names)=={str(p.relative_to(root)) for p in members}
  for p in members:assert hashlib.sha256(z.extractfile(names[str(p.relative_to(root))]).read()).hexdigest()==sha(p)
receipt_out={'status':'source_archive_NAS_scoped_restore_verified','sha256':digest,'size_bytes':receipt['size_bytes'],'nas_path':nas_path,'members':{str(p.relative_to(root)):sha(p) for p in members},'verified':True}
(card/'nas_source_archive.json').write_text(json.dumps(receipt_out,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'result':'result.json','source_archive':'nas_source_archive.json','model_fits':0,'native_accounts':0,'market_outcome_reads':0,'Qlib_Recorder':None,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]},'limitations':['Ticker identity at 2020-06-30 is not historical price coverage or exact first-public availability.','403 is this credential and date range, not proof all vendors lack delisted XLNX history.','No market bar, factor, or SOXX account result.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'reference':identity['ticker'],'aggregate_http':403,'source_nas_sha256':digest,'fits':0,'accounts':0}))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{"reference": "XLNX", "aggregate_http": 403, "source_nas_sha256": "4c147d44935904df8d3d5993586e3eea4a4f430554fd68a634fb8c85a04b3e10", "fits": 0, "accounts": 0}


# XLNX 历史 SOXX 成分股价格：Polygon 来源权限预检

冻结两个只读请求并各执行一次。Polygon/Massive 的 2020-06-30 历史 ticker 请求返回 HTTP 200、`XLNX / Xilinx Inc`、普通股身份及 CIK/FIGI；同一现有凭证请求 2019-01-02 至 2022-12-30 的拆股调整日线，返回 HTTP 403 `NOT_AUTHORIZED`，响应明确指出套餐不覆盖该历史时段。没有得到任何 XLNX 历史价格行。

这说明**当前凭证下的这条历史日线来源不可用**，不说明所有数据商或 NAS 备份都没有数据。该供应商的调整价格也不含分红调整，且当前历史响应即便可取也需另验原始身份、会话覆盖、公司行动和首次可用时间。不能用收购方 AMD 的价格回填 XLNX，更不能在缺失的 SOXX 成分股权重上重归一化。

原始 HTTP 200/403 响应、冻结计划和传输收据已经按 SHA-256 归档 Yanxuan NAS，并完成远端限定恢复。0 SOXX 收益读取、0 模型拟合、0 账户。**决定：仅关闭此凭证的 XLNX 历史价格入口；不形成因子或推广结论。** 下一步检查无需购买的其他原始历史价格来源；完整成分股覆盖和历史时点资格仍是“下跌集中度”机制的必要来源门。
